# zip_folder — 폴더 압축 유틸리티

특정 확장자를 제외하고 폴더 전체를 ZIP으로 압축합니다.  
폴더 구조는 그대로 유지됩니다.

In [ ]:
# ── [1] 파라미터 ──────────────────────────────────────────────────────────────
SRC_DIR      = r"D:\KDH\simVary\e10_6TSweep\refModel\ACLossCalcExport_Ref"
OUT_ZIP      = r""                   # 비워두면 SRC_DIR 옆에 자동 생성
EXCLUDE_EXTS = {".txt"}              # 제외할 확장자 (소문자, 복수 가능: {".txt", ".log"})
OVERWRITE    = True                  # 동일 이름 ZIP이 있을 때 덮어쓸지 여부

In [ ]:
# ── [2] 파일 목록 미리보기 ────────────────────────────────────────────────────
from pathlib import Path

src = Path(SRC_DIR)
assert src.is_dir(), f"폴더를 찾을 수 없습니다: {src}"

all_files  = list(src.rglob("*"))
all_files  = [f for f in all_files if f.is_file()]
skip_files = [f for f in all_files if f.suffix.lower() in EXCLUDE_EXTS]
zip_files  = [f for f in all_files if f.suffix.lower() not in EXCLUDE_EXTS]

print(f"전체 파일    : {len(all_files):>6}개")
print(f"제외 ({', '.join(EXCLUDE_EXTS)}): {len(skip_files):>6}개")
print(f"압축 대상    : {len(zip_files):>6}개")

# ZIP 출력 경로 결정
out_zip = Path(OUT_ZIP) if OUT_ZIP.strip() else src.parent / (src.name + "_no_txt.zip")
print(f"\n출력 ZIP: {out_zip}")
if out_zip.exists():
    print(f"  [!] 파일이 이미 존재합니다. OVERWRITE={OVERWRITE}")

In [ ]:
# ── [3] 압축 실행 ─────────────────────────────────────────────────────────────
import zipfile

if out_zip.exists() and not OVERWRITE:
    print("OVERWRITE=False 이므로 건너뜁니다.")
else:
    if out_zip.exists():
        out_zip.unlink()

    with zipfile.ZipFile(out_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for i, f in enumerate(zip_files):
            arcname = f.relative_to(src)           # 폴더 구조 유지
            zf.write(f, arcname)
            if (i + 1) % 500 == 0:
                print(f"  {i+1}/{len(zip_files)} 처리 중...", flush=True)

    size_mb = out_zip.stat().st_size / 1_048_576
    print(f"\n완료: {out_zip}")
    print(f"크기: {size_mb:.1f} MB  ({len(zip_files)}개 파일)")

In [ ]:
# ── [4] (선택) ZIP 내용 검증 ──────────────────────────────────────────────────
import zipfile

with zipfile.ZipFile(out_zip, "r") as zf:
    entries = zf.namelist()

print(f"ZIP 내 항목 수: {len(entries)}")
print("처음 10개:")
for e in entries[:10]:
    print(f"  {e}")